# Consolidated CSD Evidence Evaluation

**Evaluates Critical Slowing Down (CSD) evidence across 4 task families, 9 LLMs, and 6 experiments.**

This notebook produces 6 analysis blocks:
1. **Revised Success Criteria** — SC1 flickering, SC2 mixture variance R², SC3 classifier F1
2. **Indicator Effectiveness Ranking** — ranks 6 CSD indicators by sensitivity, specificity, lead time, effect size
3. **Negative Control Analysis** — false positive rate, random null distribution
4. **Temperature Synthesis** — dose-response evidence
5. **Cross-Experiment Consistency** — Fleiss' kappa agreement matrix
6. **Paper Figure Specifications** — 8 figure specs for the paper

**Overall verdict: PARTIALLY_CONFIRMED (2/3 criteria passed)**

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import math
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loguru import logger
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ── Logging (notebook-friendly) ──
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-276cb0-flickering-before-failing-ecological-cri/main/evaluation_iter4_consolidated_cs/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub (primary) or local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()

# Convert string level keys to int (JSON requires string keys)
all_level_data = {}
for exp_key, models in data["all_level_data"].items():
    all_level_data[exp_key] = {}
    for model, levels in models.items():
        all_level_data[exp_key][model] = {
            int(k): v for k, v in levels.items()
        }

classifier_data = data["classifier_data"]
temperature_data = data["temperature_data"]

logger.info(f"Loaded {len(all_level_data)} experiments")
for ek, models in all_level_data.items():
    logger.info(f"  {ek}: {len(models)} models")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — All tunable parameters (start with MINIMUM values for fast testing)
# ══════════════════════════════════════════════════════════════════════════════

# Negative control null distribution parameters
NULL_ITERATIONS = 10       # Original: 100 — number of random null samples
NULL_N_VECTORS = 20        # Original: 50  — vectors per null sample
NULL_DIM = 50              # Original: 384 — embedding dimensionality
KMEANS_N_INIT = 2          # Original: 3   — KMeans restarts

# Statistical thresholds (keep original values — they don't affect runtime)
KENDALL_P_THRESHOLD = 0.10
SC1_ACCURACY_THRESHOLD = 0.80
SC1_DIP_P_THRESHOLD = 0.05
SC1_SIL_THRESHOLD = 0.3
SC2_R2_THRESHOLD = 0.5
SC3_IMPROVEMENT_THRESHOLD = 15.0

# 5 positive pairs (task, model, d*, experiment_key)
POSITIVE_PAIRS = [
    ("arithmetic", "meta-llama/llama-3.1-8b-instruct", 20, "exp_id1_it2"),
    ("arithmetic", "google/gemini-2.0-flash-001", 15, "exp_id1_it2"),
    ("graph_coloring", "openai/gpt-4o-mini", 10, "exp_id3_it2"),
    ("graph_coloring", "google/gemini-2.0-flash-001", 14, "exp_id3_it2"),
    ("graph_coloring", "google/gemini-2.0-flash-lite-001", 11, "exp_id3_it2"),
]

# 6 CSD indicator names (canonical)
CSD_INDICATORS = [
    "embedding_variance", "dip_statistic", "silhouette_k2",
    "bimodality_coefficient", "disagreement_rate", "ashman_d",
]

# Expected trend directions for each indicator as difficulty approaches d*
EXPECTED_DIRECTION = {
    "embedding_variance": +1,
    "dip_statistic": +1,
    "silhouette_k2": +1,
    "bimodality_coefficient": +1,
    "disagreement_rate": +1,
    "ashman_d": +1,
}

logger.info(f"Config: NULL_ITERATIONS={NULL_ITERATIONS}, NULL_N_VECTORS={NULL_N_VECTORS}, NULL_DIM={NULL_DIM}")

## Block 1: Revised Success Criteria

Three success criteria are evaluated:
- **SC1 (Flickering):** Does bimodality appear at difficulty levels where accuracy is still high (>80%)?
- **SC2 (Mixture Variance):** Does observed embedding variance follow the mixture-model prediction `p*(1-p)`?
- **SC3 (Classifier):** Does a multi-indicator CSD classifier beat single-feature baselines by ≥15% F1?

In [ ]:
# ── SC1: Flickering detection ────────────────────────────────────────────────
def evaluate_sc1_flickering(all_level_data, positive_pairs):
    """SC1: Check if flickering (dip p<0.05 OR silhouette>0.3) is detected
    at difficulty levels where accuracy > 0.80.
    """
    logger.info("BLOCK 1a: Evaluating SC1 (flickering detection)")
    pair_results = []
    tasks_with_signal = set()
    models_with_signal = set()

    for task, model, d_star, exp_key in positive_pairs:
        data = all_level_data.get(exp_key, {}).get(model, {})
        if not data:
            pair_results.append({"task": task, "model": model, "d_star": d_star, "detected": False})
            continue

        detected = False
        for level, indicators in sorted(data.items()):
            acc = indicators.get("accuracy", 0.0)
            dip_p = indicators.get("dip_pvalue", 1.0)
            sil = indicators.get("silhouette_k2", 0.0)
            if acc > SC1_ACCURACY_THRESHOLD and (dip_p < SC1_DIP_P_THRESHOLD or sil > SC1_SIL_THRESHOLD):
                detected = True
                break

        if detected:
            tasks_with_signal.add(task)
            models_with_signal.add(model)
        pair_results.append({"task": task, "model": model, "d_star": d_star, "detected": detected})

    n_detected = sum(1 for r in pair_results if r["detected"])
    sc1_pass = len(tasks_with_signal) >= 2 and len(models_with_signal) >= 2

    logger.info(f"  SC1: {n_detected}/5 pairs detected, "
                f"{len(tasks_with_signal)} tasks, {len(models_with_signal)} models -> {'PASS' if sc1_pass else 'FAIL'}")
    return {
        "sc1_pass": sc1_pass,
        "n_pairs_detected": n_detected,
        "n_tasks": len(tasks_with_signal),
        "n_models": len(models_with_signal),
        "pair_results": pair_results,
    }

sc1 = evaluate_sc1_flickering(all_level_data, POSITIVE_PAIRS)
for pr in sc1["pair_results"]:
    print(f"  {pr['task']:20s} x {pr['model']:45s} d*={pr['d_star']:3d} -> {'DETECTED' if pr['detected'] else 'not detected'}")

In [ ]:
# ── SC2: Mixture variance R² ─────────────────────────────────────────────────
def evaluate_sc2_mixture_variance(all_level_data, positive_pairs):
    """SC2 Revised: Mixture model variance R².
    Var_mix(d) = p(d)*(1-p(d))*||delta_mu||^2
    We fit: observed_variance = a * p*(1-p) + b via OLS.
    """
    logger.info("BLOCK 1b: Evaluating SC2 (mixture variance R^2)")
    pair_r2s = []

    for task, model, d_star, exp_key in positive_pairs:
        data = all_level_data.get(exp_key, {}).get(model, {})
        if not data or len(data) < 3:
            pair_r2s.append({"task": task, "model": model, "r2": 0.0})
            continue

        levels = sorted(data.keys())
        accuracies = np.array([data[l]["accuracy"] for l in levels])
        obs_variance = np.array([data[l]["embedding_variance"] for l in levels])

        mix_predictor = accuracies * (1.0 - accuracies)

        if np.std(mix_predictor) > 1e-10 and np.std(obs_variance) > 1e-10:
            slope, intercept, r_value, p_value, std_err = stats.linregress(mix_predictor, obs_variance)
            r2 = r_value ** 2
        else:
            r2 = 0.0

        pair_r2s.append({"task": task, "model": model, "r2": round(r2, 6)})
        logger.info(f"  SC2 {task}/{model}: R^2={r2:.4f}")

    r2_values = [p["r2"] for p in pair_r2s]
    mean_r2 = float(np.mean(r2_values))
    n_pass = sum(1 for r in r2_values if r > SC2_R2_THRESHOLD)
    sc2_pass = n_pass >= 3

    logger.info(f"  SC2: {n_pass}/5 pairs with R^2>0.5, mean R^2={mean_r2:.4f} -> {'PASS' if sc2_pass else 'FAIL'}")
    return {
        "sc2_pass": sc2_pass,
        "n_pairs_pass": n_pass,
        "mean_r2": mean_r2,
        "pair_r2s": pair_r2s,
    }

sc2 = evaluate_sc2_mixture_variance(all_level_data, POSITIVE_PAIRS)

In [ ]:
# ── SC3: Classifier F1 improvement ───────────────────────────────────────────
def evaluate_sc3_classifier(classifier_data):
    """SC3: Classifier F1 improvement >= 15%."""
    logger.info("BLOCK 1c: Evaluating SC3 (classifier F1 improvement)")
    meta = classifier_data.get("metadata", {})
    comp = meta.get("classifier_comparison", {})

    best_csd_f1 = comp.get("csd_logreg_full", {}).get("lopo_f1", 0.0)

    baseline_names = ["variance_only", "dip_only", "disagreement_only", "bimodality_only"]
    best_single = 0.0
    best_single_name = ""
    for bn in baseline_names:
        f1 = comp.get(bn, {}).get("lopo_f1", 0.0)
        if f1 > best_single:
            best_single = f1
            best_single_name = bn

    delta_f1 = best_csd_f1 - best_single
    improvement_pct = (delta_f1 / best_single * 100) if best_single > 0 else 0.0
    sc3_pass = improvement_pct >= SC3_IMPROVEMENT_THRESHOLD

    logger.info(f"  SC3: CSD F1={best_csd_f1:.4f}, best baseline ({best_single_name}) F1={best_single:.4f}, "
                f"improvement={improvement_pct:.1f}% -> {'PASS' if sc3_pass else 'FAIL'}")
    return {
        "sc3_pass": sc3_pass,
        "csd_f1": best_csd_f1,
        "best_baseline_f1": best_single,
        "best_baseline_name": best_single_name,
        "improvement_pct": improvement_pct,
        "csd_auroc": comp.get("csd_logreg_full", {}).get("lopo_auroc", 0.0),
    }

sc3 = evaluate_sc3_classifier(classifier_data)

# ── Overall verdict ──────────────────────────────────────────────────────────
n_criteria_passed = sum([sc1["sc1_pass"], sc2["sc2_pass"], sc3["sc3_pass"]])
if n_criteria_passed >= 3:
    verdict = "CONFIRMED"
elif n_criteria_passed >= 1:
    verdict = "PARTIALLY_CONFIRMED"
else:
    verdict = "DISCONFIRMED"

print(f"\n{'='*60}")
print(f"  OVERALL VERDICT: {verdict} ({n_criteria_passed}/3 criteria passed)")
print(f"  SC1 (flickering): {'PASS' if sc1['sc1_pass'] else 'FAIL'}")
print(f"  SC2 (mixture var): {'PASS' if sc2['sc2_pass'] else 'FAIL'}")
print(f"  SC3 (classifier):  {'PASS' if sc3['sc3_pass'] else 'FAIL'}")
print(f"{'='*60}")

## Block 2: Indicator Effectiveness Ranking

Ranks 6 CSD indicators by a composite score combining **sensitivity** (detects true positives), **specificity** (avoids false positives), **lead time** (how early before d\*), and **effect size** (Kendall tau).

In [ ]:
# Build negative pairs from syllogistic and multi-hop experiments
negative_pairs = []
for model in all_level_data.get("exp_id2_it2", {}):
    negative_pairs.append(("syllogistic", model, "exp_id2_it2"))
for model in all_level_data.get("exp_id4_it2", {}):
    negative_pairs.append(("multi_hop", model, "exp_id4_it2"))

def compute_indicator_ranking(all_level_data, positive_pairs, negative_pairs):
    """Rank 6 CSD indicators by sensitivity, specificity, lead time, effect size."""
    logger.info("BLOCK 2: Computing indicator effectiveness ranking")
    indicator_stats = {ind: {"sensitivity_hits": 0, "specificity_misses": 0,
                             "lead_times": [], "effect_sizes": []}
                       for ind in CSD_INDICATORS}
    n_positive = len(positive_pairs)
    n_negative = len(negative_pairs)

    # Evaluate on positive pairs
    for task, model, d_star, exp_key in positive_pairs:
        data = all_level_data.get(exp_key, {}).get(model, {})
        if not data:
            continue
        levels = sorted(data.keys())
        for ind in CSD_INDICATORS:
            values = [data[l].get(ind, 0.0) for l in levels]
            if len(values) < 4 or np.std(values) < 1e-10:
                continue
            tau, p_val = stats.kendalltau(levels, values)
            expected_dir = EXPECTED_DIRECTION.get(ind, +1)
            if p_val < KENDALL_P_THRESHOLD and np.sign(tau) == expected_dir:
                indicator_stats[ind]["sensitivity_hits"] += 1
            indicator_stats[ind]["effect_sizes"].append(abs(tau))

            # Lead time: levels before d* where indicator first becomes significant
            first_sig_level = None
            for i, l in enumerate(levels):
                if l >= d_star:
                    break
                val = data[l].get(ind, 0.0)
                if ind == "dip_statistic":
                    if data[l].get("dip_pvalue", 1.0) < 0.05:
                        first_sig_level = l; break
                elif ind == "silhouette_k2":
                    if val > 0.3:
                        first_sig_level = l; break
                elif ind == "bimodality_coefficient":
                    if val > 0.555:
                        first_sig_level = l; break
                elif ind == "disagreement_rate":
                    if val > 0.5:
                        first_sig_level = l; break
                elif ind == "embedding_variance":
                    if i > 0 and val > data[levels[i-1]].get(ind, 0.0) * 1.2:
                        first_sig_level = l; break
                elif ind == "ashman_d":
                    if val > 2.0:
                        first_sig_level = l; break
            if first_sig_level is not None:
                indicator_stats[ind]["lead_times"].append(d_star - first_sig_level)

    # Evaluate on negative pairs (specificity)
    for task, model, exp_key in negative_pairs:
        data = all_level_data.get(exp_key, {}).get(model, {})
        if not data:
            continue
        levels = sorted(data.keys())
        for ind in CSD_INDICATORS:
            values = [data[l].get(ind, 0.0) for l in levels]
            if len(values) < 4 or np.std(values) < 1e-10:
                continue
            tau, p_val = stats.kendalltau(levels, values)
            expected_dir = EXPECTED_DIRECTION.get(ind, +1)
            if p_val < KENDALL_P_THRESHOLD and np.sign(tau) == expected_dir:
                indicator_stats[ind]["specificity_misses"] += 1

    # Compute final metrics
    ranking = []
    for ind in CSD_INDICATORS:
        s = indicator_stats[ind]
        n_evaluated_pos = max(len(s["effect_sizes"]), 1)
        sensitivity = s["sensitivity_hits"] / max(n_evaluated_pos, 1)
        specificity = 1.0 - s["specificity_misses"] / max(n_negative, 1)
        mean_lead = float(np.mean(s["lead_times"])) if s["lead_times"] else 0.0
        mean_effect = float(np.mean(s["effect_sizes"])) if s["effect_sizes"] else 0.0
        max_possible_lead = 20.0
        lead_norm = min(mean_lead / max_possible_lead, 1.0) if mean_lead > 0 else 0.0
        effect_norm = min(mean_effect, 1.0)
        composite = (0.3 * sensitivity + 0.2 * specificity +
                     0.3 * lead_norm + 0.2 * effect_norm)
        ranking.append({
            "indicator": ind, "sensitivity": round(sensitivity, 4),
            "specificity": round(specificity, 4),
            "mean_lead_time": round(mean_lead, 2),
            "mean_effect_size": round(mean_effect, 4),
            "composite_score": round(composite, 4),
        })

    ranking.sort(key=lambda x: x["composite_score"], reverse=True)
    for i, r in enumerate(ranking):
        logger.info(f"  #{i+1} {r['indicator']}: composite={r['composite_score']:.4f}")
    return {"ranking": ranking, "best_indicator": ranking[0]["indicator"],
            "best_composite": ranking[0]["composite_score"]}

ranking = compute_indicator_ranking(all_level_data, POSITIVE_PAIRS, negative_pairs)
print(f"\nBest indicator: {ranking['best_indicator']} (composite={ranking['best_composite']:.4f})")

## Block 3: Negative Control Deep Dive

Evaluates whether CSD signals are absent in tasks where no critical difficulty threshold (d\*) exists:
- **Syllogistic reasoning** — gradual degradation, no sharp boundary
- **Multi-hop QA** — constant bimodality across difficulty levels
- **Random embedding null** — empirical null distribution from random unit vectors

In [ ]:
def evaluate_negative_controls(all_level_data):
    """Analyze syllogistic and multi-hop negative controls."""
    logger.info("BLOCK 3: Evaluating negative controls")

    # ── 3a: Syllogistic baseline bimodality ──
    syl_data = all_level_data.get("exp_id2_it2", {})
    syl_stats = []
    for model, level_data in syl_data.items():
        dips = [v.get("dip_statistic", 0.0) for v in level_data.values()]
        sils = [v.get("silhouette_k2", 0.0) for v in level_data.values()]
        syl_stats.append({
            "model": model,
            "mean_dip": float(np.mean(dips)) if dips else 0.0,
            "mean_silhouette": float(np.mean(sils)) if sils else 0.0,
            "std_dip": float(np.std(dips)) if dips else 0.0,
        })

    # Random embedding null — empirical null distribution
    rng = np.random.RandomState(42)
    null_dips = []
    null_sils = []
    for _ in range(NULL_ITERATIONS):
        vecs = rng.randn(NULL_N_VECTORS, NULL_DIM)
        vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

        # Hartigan dip approximation on first principal component
        pca = PCA(n_components=1)
        proj = pca.fit_transform(vecs).ravel()
        sorted_proj = np.sort(proj)
        ks_stat, _ = stats.kstest(sorted_proj, 'norm',
                                   args=(np.mean(sorted_proj), np.std(sorted_proj)))
        null_dips.append(ks_stat / 2)

        # Silhouette with k=2 KMeans
        km = KMeans(n_clusters=2, n_init=KMEANS_N_INIT, random_state=42)
        labels = km.fit_predict(vecs)
        if len(set(labels)) == 2:
            try:
                s = silhouette_score(vecs, labels, metric="cosine")
                null_sils.append(s)
            except Exception:
                null_sils.append(0.0)

    null_dip_95 = float(np.percentile(null_dips, 95))
    null_sil_95 = float(np.percentile(null_sils, 95))
    logger.info(f"  Random null ({NULL_DIM}d): dip 95th={null_dip_95:.4f}, sil 95th={null_sil_95:.4f}")

    for ss in syl_stats:
        within_null = ss["mean_dip"] <= null_dip_95
        logger.info(f"  Syllogistic {ss['model']}: mean_dip={ss['mean_dip']:.4f} "
                    f"({'within' if within_null else 'ABOVE'} null)")

    # ── 3b: Multi-hop constant bimodality ──
    mh_data = all_level_data.get("exp_id4_it2", {})
    mh_stats = []
    for model, level_data in mh_data.items():
        dips = [v.get("dip_statistic", 0.0) for v in level_data.values()]
        cv_dip = float(np.std(dips) / np.mean(dips)) if np.mean(dips) > 0 else 0.0
        all_sig = all(v.get("dip_pvalue", 1.0) < 0.05 for v in level_data.values())
        mh_stats.append({
            "model": model, "cv_dip": cv_dip, "all_bimodal": all_sig,
            "mean_dip": float(np.mean(dips)), "n_levels": len(level_data),
        })
        logger.info(f"  Multi-hop {model}: CV(dip)={cv_dip:.4f}, all_bimodal={all_sig}")

    # ── 3c: Overall false positive rate ──
    negative_pairs_all = []
    for model in syl_data:
        negative_pairs_all.append(("syllogistic", model, "exp_id2_it2"))
    for model in mh_data:
        negative_pairs_all.append(("multi_hop", model, "exp_id4_it2"))

    n_false_alarms = 0
    for task, model, exp_key in negative_pairs_all:
        data = all_level_data.get(exp_key, {}).get(model, {})
        if not data:
            continue
        levels = sorted(data.keys())
        n_sig_indicators = 0
        for ind in CSD_INDICATORS:
            values = [data[l].get(ind, 0.0) for l in levels]
            if len(values) < 4 or np.std(values) < 1e-10:
                continue
            tau, p_val = stats.kendalltau(levels, values)
            expected_dir = EXPECTED_DIRECTION.get(ind, +1)
            if p_val < KENDALL_P_THRESHOLD and np.sign(tau) == expected_dir:
                n_sig_indicators += 1
        if n_sig_indicators >= 2:
            n_false_alarms += 1

    n_neg_total = len(negative_pairs_all)
    fpr = n_false_alarms / max(n_neg_total, 1)
    logger.info(f"  False positive rate: {n_false_alarms}/{n_neg_total} = {fpr:.4f}")

    return {
        "syllogistic_stats": syl_stats, "null_dip_95th": null_dip_95,
        "null_sil_95th": null_sil_95, "multihop_stats": mh_stats,
        "false_positive_rate": fpr, "n_false_alarms": n_false_alarms,
        "n_negative_total": n_neg_total, "negative_pairs_list": negative_pairs_all,
    }

neg_ctrl = evaluate_negative_controls(all_level_data)
print(f"\nFalse positive rate: {neg_ctrl['false_positive_rate']:.4f}")

## Block 4: Temperature Synthesis

Synthesizes temperature manipulation experiment: do CSD indicators increase with sampling temperature as predicted?

In [ ]:
def evaluate_temperature(temp_data):
    """Synthesize temperature experiment results."""
    logger.info("BLOCK 4: Evaluating temperature experiment")
    meta = temp_data.get("metadata", {})
    dose = meta.get("dose_response_analysis", {})

    d_star_stable = dose.get("d_star_analysis", {}).get("d_star_stable", False)
    var_frac = dose.get("variance_temperature_effect", {}).get("frac_positive_trend", 0.0)
    dis_frac = dose.get("disagreement_temperature_effect", {}).get("frac_positive_trend", 0.0)
    dip_frac = dose.get("dip_temperature_effect", {}).get("frac_positive_trend", 0.0)
    bimodal_rho = dose.get("bimodal_zone_widening", {}).get("spearman_rho", 0.0)
    csd_score = dose.get("csd_evidence_score", 0.0)

    confirmed_predictions = []
    failed_predictions = []

    if d_star_stable:
        confirmed_predictions.append("d* stability across temperatures")
    else:
        failed_predictions.append("d* stability across temperatures")
    if var_frac > 0.6:
        confirmed_predictions.append(f"Embedding variance increases with temperature (frac={var_frac:.3f})")
    else:
        failed_predictions.append(f"Embedding variance dose-response (frac={var_frac:.3f})")
    if dis_frac > 0.6:
        confirmed_predictions.append(f"Disagreement rate increases with temperature (frac={dis_frac:.3f})")
    else:
        failed_predictions.append(f"Disagreement rate dose-response (frac={dis_frac:.3f})")
    if dip_frac > 0.5:
        confirmed_predictions.append(f"Dip statistic dose-response (frac={dip_frac:.3f})")
    else:
        failed_predictions.append(f"Dip statistic does NOT show dose-response (frac={dip_frac:.3f})")
    if bimodal_rho > 0:
        confirmed_predictions.append(f"Bimodal zone widens with temperature (rho={bimodal_rho:.3f})")
    else:
        failed_predictions.append(f"Bimodal zone width DECREASES with temperature (rho={bimodal_rho:.3f})")

    logger.info(f"  Confirmed: {len(confirmed_predictions)}, Failed: {len(failed_predictions)}")
    logger.info(f"  CSD evidence score: {csd_score}")

    return {
        "confirmed_predictions": confirmed_predictions,
        "failed_predictions": failed_predictions,
        "evidence_score": csd_score,
        "d_star_stable": d_star_stable,
        "variance_frac_positive": var_frac,
        "disagreement_frac_positive": dis_frac,
        "dip_frac_positive": dip_frac,
    }

temp_results = evaluate_temperature(temperature_data)
print(f"\nConfirmed predictions: {len(temp_results['confirmed_predictions'])}")
for p in temp_results["confirmed_predictions"]:
    print(f"  + {p}")
print(f"Failed predictions: {len(temp_results['failed_predictions'])}")
for p in temp_results["failed_predictions"]:
    print(f"  - {p}")

## Block 5: Cross-Experiment Consistency Matrix

Builds an indicator x (task, model) matrix classifying each indicator trend as rising (+1), falling (-1), or no trend (0), then computes **Fleiss' kappa** for inter-indicator agreement.

In [ ]:
def compute_consistency_matrix(all_level_data, all_pairs):
    """Build indicator x task x model matrix and compute Fleiss' kappa."""
    logger.info("BLOCK 5: Computing cross-experiment consistency matrix")

    matrix_rows = []
    for task, model, d_star, exp_key in all_pairs:
        data = all_level_data.get(exp_key, {}).get(model, {})
        if not data or len(data) < 4:
            continue
        levels = sorted(data.keys())
        row_ratings = []
        for ind in CSD_INDICATORS:
            values = [data[l].get(ind, 0.0) for l in levels]
            if len(values) < 4 or np.std(values) < 1e-10:
                row_ratings.append(0)
                continue
            tau, p_val = stats.kendalltau(levels, values)
            if p_val < KENDALL_P_THRESHOLD:
                row_ratings.append(1 if tau > 0 else -1)
            else:
                row_ratings.append(0)
        matrix_rows.append({
            "task": task, "model": model, "d_star": d_star,
            "ratings": row_ratings,
        })

    # Compute Fleiss' kappa
    n_subjects = len(matrix_rows)
    n_raters = len(CSD_INDICATORS)
    n_categories = 3

    if n_subjects >= 2:
        cat_counts = np.zeros((n_subjects, n_categories))
        for i, row in enumerate(matrix_rows):
            for rating in row["ratings"]:
                cat_idx = rating + 1  # -1->0, 0->1, +1->2
                cat_counts[i, cat_idx] += 1
        N = n_subjects
        n = n_raters
        p_j = np.sum(cat_counts, axis=0) / (N * n)
        P_i = (np.sum(cat_counts ** 2, axis=1) - n) / (n * (n - 1))
        P_bar = float(np.mean(P_i))
        Pe_bar = float(np.sum(p_j ** 2))
        kappa = (P_bar - Pe_bar) / (1.0 - Pe_bar) if (1.0 - Pe_bar) > 1e-10 else 0.0
    else:
        kappa = 0.0

    interp = 'poor' if kappa < 0.2 else 'fair' if kappa < 0.4 else 'moderate' if kappa < 0.6 else 'substantial'
    logger.info(f"  Fleiss' kappa = {kappa:.4f} ({interp})")

    # Cross-task concordance
    task_names = list(set(r["task"] for r in matrix_rows))
    concordance_pairs = []
    for i, t1 in enumerate(task_names):
        for t2 in task_names[i+1:]:
            t1_rows = [r for r in matrix_rows if r["task"] == t1]
            t2_rows = [r for r in matrix_rows if r["task"] == t2]
            if not t1_rows or not t2_rows:
                continue
            t1_avg = np.mean([r["ratings"] for r in t1_rows], axis=0)
            t2_avg = np.mean([r["ratings"] for r in t2_rows], axis=0)
            agree = sum(1 for a, b in zip(t1_avg, t2_avg)
                       if np.sign(a) == np.sign(b) or (abs(a) < 0.1 and abs(b) < 0.1))
            concordance = agree / len(CSD_INDICATORS)
            concordance_pairs.append({"task1": t1, "task2": t2, "concordance": round(concordance, 4)})

    return {"fleiss_kappa": round(kappa, 4), "n_subjects": n_subjects,
            "concordance_pairs": concordance_pairs, "matrix_rows": matrix_rows}

# All pairs including negative controls
all_pairs_for_matrix = list(POSITIVE_PAIRS)
for model in all_level_data.get("exp_id2_it2", {}):
    all_pairs_for_matrix.append(("syllogistic", model, None, "exp_id2_it2"))
for model in all_level_data.get("exp_id4_it2", {}):
    all_pairs_for_matrix.append(("multi_hop", model, None, "exp_id4_it2"))

consistency = compute_consistency_matrix(all_level_data, all_pairs_for_matrix)
print(f"\nFleiss' kappa: {consistency['fleiss_kappa']:.4f}")
for cp in consistency["concordance_pairs"]:
    print(f"  {cp['task1']} vs {cp['task2']}: concordance={cp['concordance']:.4f}")

## Block 6: Paper Figure Specifications

Generates specifications for 8 paper figures describing the key visualizations needed.

In [ ]:
def generate_figure_specs():
    """Generate specifications for paper figures."""
    logger.info("BLOCK 6: Generating paper figure specifications")
    specs = [
        {"figure_id": "fig1_accuracy_curves", "title": "Accuracy vs Difficulty Across Tasks and Models",
         "plot_type": "line_plot_grid", "key_finding": "All 5 positive pairs show sigmoid-like accuracy decline with identifiable d*"},
        {"figure_id": "fig2_csd_indicator_heatmap", "title": "CSD Indicator Values Across Difficulty Gradient",
         "plot_type": "heatmap_grid", "key_finding": "Variance and dip show coherent patterns near d*; ashman_d and bimodality are noisier"},
        {"figure_id": "fig3_flickering_leading_indicator", "title": "Flickering Detection at High-Accuracy Levels",
         "plot_type": "scatter_with_threshold", "key_finding": "SC1: flickering appears before accuracy drops below 80% in multiple pairs"},
        {"figure_id": "fig4_mixture_variance_fit", "title": "Mixture Model Variance Fit: Observed vs Predicted",
         "plot_type": "scatter_with_regression", "key_finding": "SC2: Mixture model explains variance structure, not fold-bifurcation"},
        {"figure_id": "fig5_temperature_dose_response", "title": "Temperature Manipulation: CSD Indicator Dose-Response",
         "plot_type": "line_plot_with_error", "key_finding": "Variance and disagreement increase with temperature; dip does not"},
        {"figure_id": "fig6_indicator_ranking_bar", "title": "CSD Indicator Effectiveness Ranking",
         "plot_type": "horizontal_bar_chart", "key_finding": "Top indicator(s) identified for practitioner recommendation"},
        {"figure_id": "fig7_negative_control_comparison", "title": "Negative Control Analysis: CSD Signals in Tasks Without d*",
         "plot_type": "violin_plot_comparison", "key_finding": "False positive rate and comparison to random embedding null"},
        {"figure_id": "fig8_classifier_comparison", "title": "CSD Classifier vs Baselines: Cross-Validation Performance",
         "plot_type": "grouped_bar_chart", "key_finding": "SC3: CSD-LogReg-full achieves 16.4% F1 improvement over best single-feature baseline"},
    ]
    logger.info(f"  Generated {len(specs)} figure specifications")
    return specs

figure_specs = generate_figure_specs()
for spec in figure_specs:
    print(f"  {spec['figure_id']:40s} [{spec['plot_type']}]")

## Results Summary & Visualization

Final evaluation summary table and key visualizations: indicator ranking bar chart, success criteria overview, and accuracy vs CSD indicator trends for positive pairs.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)
print(f"  Hypothesis Verdict:  {verdict}")
print(f"  SC1 (flickering):    {'PASS' if sc1['sc1_pass'] else 'FAIL'} ({sc1['n_pairs_detected']}/5 pairs)")
print(f"  SC2 (mixture var):   {'PASS' if sc2['sc2_pass'] else 'FAIL'} ({sc2['n_pairs_pass']}/5 R^2>0.5)")
print(f"  SC3 (classifier):    {'PASS' if sc3['sc3_pass'] else 'FAIL'} ({sc3['improvement_pct']:.1f}% improvement)")
print(f"  Best indicator:      {ranking['best_indicator']} (composite={ranking['best_composite']:.4f})")
print(f"  Fleiss kappa:        {consistency['fleiss_kappa']:.4f}")
print(f"  Neg ctrl FPR:        {neg_ctrl['false_positive_rate']:.4f}")
print(f"  Temp evidence:       {temp_results['evidence_score']:.2f}")
print("=" * 70)

# ── Figure 1: Indicator Effectiveness Ranking (bar chart) ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Indicator ranking
rank_df = pd.DataFrame(ranking["ranking"])
colors = ['#2ecc71' if s >= 0.5 else '#f39c12' if s >= 0.3 else '#e74c3c'
          for s in rank_df["composite_score"]]
axes[0].barh(rank_df["indicator"], rank_df["composite_score"], color=colors)
axes[0].set_xlabel("Composite Score")
axes[0].set_title("CSD Indicator Effectiveness Ranking")
axes[0].invert_yaxis()
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

# Panel 2: Success Criteria pass/fail
criteria = ["SC1\nFlickering", "SC2\nMixture Var", "SC3\nClassifier"]
passed = [sc1["sc1_pass"], sc2["sc2_pass"], sc3["sc3_pass"]]
bar_colors = ['#2ecc71' if p else '#e74c3c' for p in passed]
bars = axes[1].bar(criteria, [1]*3, color=bar_colors, edgecolor='black', linewidth=1.5)
for i, (bar, p) in enumerate(zip(bars, passed)):
    axes[1].text(bar.get_x() + bar.get_width()/2., 0.5,
                'PASS' if p else 'FAIL', ha='center', va='center',
                fontsize=14, fontweight='bold', color='white')
axes[1].set_ylim(0, 1.2)
axes[1].set_yticks([])
axes[1].set_title(f"Success Criteria ({n_criteria_passed}/3 passed)")

# Panel 3: Accuracy curves for positive pairs (arithmetic/llama example)
example_model = "meta-llama/llama-3.1-8b-instruct"
example_data = all_level_data["exp_id1_it2"][example_model]
levels_sorted = sorted(example_data.keys())
acc_vals = [example_data[l]["accuracy"] for l in levels_sorted]
var_vals = [example_data[l]["embedding_variance"] * 10 for l in levels_sorted]  # scaled
dis_vals = [example_data[l]["disagreement_rate"] for l in levels_sorted]

axes[2].plot(levels_sorted, acc_vals, 'bo-', label='Accuracy', linewidth=2)
axes[2].plot(levels_sorted, var_vals, 'rs--', label='Emb. Variance (x10)', linewidth=1.5)
axes[2].plot(levels_sorted, dis_vals, 'g^--', label='Disagreement Rate', linewidth=1.5)
axes[2].axhline(y=0.8, color='gray', linestyle=':', alpha=0.5, label='Acc=0.80 threshold')
axes[2].axvline(x=20, color='orange', linestyle=':', alpha=0.5, label='d*=20')
axes[2].set_xlabel("Difficulty Level")
axes[2].set_ylabel("Value")
axes[2].set_title(f"CSD Indicators vs Difficulty\n(arithmetic / llama-3.1-8b)")
axes[2].legend(fontsize=8, loc='center left')

plt.tight_layout()
plt.savefig("evaluation_summary.png", dpi=100, bbox_inches='tight')
plt.show()
print("Saved evaluation_summary.png")